Sketch: Otaku Engine Exploratory Analysis
Project: MAL-Analytics

Objective: Test the integrity of the dbt Marts and discover the "stories" hidden in the data before building the final report.

Initializing the environment, connecting to the DuckDB database, and verifying that the Tables (Marts) are accessible.

In [1]:
import duckdb
import pandas as pd
import plotly.express as px # Great for interactive charts

# Connect to the data warehouse
con = duckdb.connect('../data/mal_data.duckdb')

# Verify table presence
con.sql("SHOW TABLES").df()

,name
0,dim_anime
1,dim_demographics
2,dim_genres
3,dim_studios
4,dim_themes
5,fct_anime_stats
6,raw_anime_2026_03_02
7,stg_anime


Tables are visible so we can begin some analysis.

High-level summary: First some queries to get some general stats and overview. 

In [3]:
res_overview = con.sql("""
    SELECT
        count(anime_id) as total_titles,
        round(avg(score), 2) as global_avg_score,
        sum(scored_by) as total_votes
    FROM fct_anime_stats
""").df()

res_overview

,total_titles,global_avg_score,total_votes
0,2500,7.85,373199618.0


Connection works as intended. Values are being properly received and we have the intended number of titles. Moving on.

-Total Averages: Exploring the total averages of the dataset.

In [6]:
res_averages = con.sql("""
    SELECT
        ROUND(AVG(members), 0) as avg_members,
        ROUND(AVG(favorites), 0) as avg_favorites,
        ROUND(100.0 * AVG(favorites) / NULLIF(AVG(members), 0), 2) as favorite_conversion_pct,
        ROUND(AVG(scored_by), 0) as avg_voters,
        ROUND(100.0 * AVG(scored_by) / NULLIF(AVG(members), 0), 2) as voter_conversion_pct
    FROM fct_anime_stats
""").df()

res_averages

,avg_members,avg_favorites,favorite_conversion_pct,avg_voters,voter_conversion_pct
0,278069.0,4381.0,1.58,149280.0,53.68


This table gives us some interesting numbers regarding the top 2500. Only about 1.6% of all the people that list an anime (members) actually favorite them, and about half of the people put in the effort to vote them.

In [10]:
queryText = """
    SELECT
        a.title_default,
        s.score,
        s.rank,
        s.popularity,
        s.members,
        s.favorites,
        ROUND(100.0 * s.favorites / NULLIF(s.members, 0), 2) as loyalty_ratio
    FROM fct_anime_stats s
    JOIN dim_anime a ON s.anime_id = a.anime_id
    WHERE s.members > 10000
    ORDER BY loyalty_ratio DESC  
"""

df_edges = con.sql(queryText).df()

display("Top 5:", df_edges.head(5))
display("Bottom 5:", df_edges.tail(5))

'Top 5:'

,title_default,score,rank,popularity,members,favorites,loyalty_ratio
0,One Piece,8.73,52,17,2642204,249119,9.43
1,Hunter x Hunter (2011),9.03,9,8,3149760,227703,7.23
2,Steins;Gate,9.07,5,14,2785152,201087,7.22
3,Fullmetal Alchemist: Brotherhood,9.11,3,3,3643577,240758,6.61
4,Sousou no Frieren,9.28,1,109,1349519,85647,6.35


'Bottom 5:'

,title_default,score,rank,popularity,members,favorites,loyalty_ratio
2247,Minami-ke Betsubara,7.48,2174,4566,30663,10,0.03
2248,Hidamari Sketch Specials,7.51,2039,5245,22126,7,0.03
2249,Meitantei Conan OVA 04: Conan to Kid to Crysta...,7.49,2124,5578,18881,3,0.02
2250,Kamichu! Specials,7.45,2287,6531,12628,3,0.02
2251,Meitantei Conan Movie 09: Suihei Senjou no Str...,7.61,1649,7058,10476,2,0.02


A thought; the bottom 5 are all a non-main franchise media type. They're all either OVA/Movie/Specials. Detective Conan is a highly praised and famous series but since MAL separate them by entries it skews the results; One may favorite the main series entry but ignore Movie 09.
Therefore, I will do a quick review of media type stats:

In [11]:
query_type_loyalty = """
SELECT 
    a.media_type,
    COUNT(a.anime_id) as total_titles,
    ROUND(AVG(100.0 * s.favorites / NULLIF(s.members, 0)), 2) as avg_loyalty_ratio,
    ROUND(AVG(s.score), 2) as avg_score
FROM dim_anime a
JOIN fct_anime_stats s ON a.anime_id = s.anime_id
GROUP BY 1
ORDER BY avg_loyalty_ratio DESC
"""

df_type_loyalty = con.sql(query_type_loyalty).df()
df_type_loyalty

,media_type,total_titles,avg_loyalty_ratio,avg_score
0,TV,1216,1.12,7.90
1,PV,11,0.87,7.79
2,ONA,270,0.78,7.80
3,Music,55,0.60,7.70
4,OVA,240,0.54,7.75
5,Movie,526,0.52,7.87
6,CM,4,0.45,7.64
7,TV Special,64,0.31,7.86
8,Special,120,0.16,7.71


TV is expectedly the main media has a higher adherence ratio compared to others; which are often related to the main media. One may watch the special, but may not always favorite the special of an anime they already favorited the main series.
Also very interesting to notice how all these averages are lower than the global average (1.58). This means some anime are so massive (One Piece as an example), that they alone can move the global average.

This got me curious to see what are the Genres with highest loyalty ratio.

In [15]:
queryText = """
    SELECT
        g.genre_name,
        COUNT(g.anime_id) as total_titles,
        ROUND(AVG(100.0 * s.favorites / NULLIF(s.members, 0)), 2) as avg_loyalty_ratio,
        ROUND(AVG(s.score), 2) as avg_score
    FROM dim_genres g
    JOIN fct_anime_stats s ON g.anime_id = s.anime_id
    GROUP BY 1
    HAVING total_titles > 20
    ORDER BY avg_loyalty_ratio DESC
"""

df_genre_loyalty = con.sql(queryText).df()
df_genre_loyalty.head(10)

,genre_name,total_titles,avg_loyalty_ratio,avg_score
0,Award Winning,118,1.45,8.06
1,Suspense,152,1.27,8.01
2,Hentai,30,1.26,7.60
3,Horror,51,1.08,7.82
4,Drama,649,1.08,7.93
5,Sci-Fi,381,0.94,7.87
6,Romance,443,0.88,7.82
7,Adventure,578,0.86,7.85
8,Mystery,278,0.84,7.94
9,Boys Love,35,0.84,7.77


As the last sketch here, I will do a quick check on the Studios.

In [17]:
queryText = """
    SELECT
        s.studio,
        COUNT(s.anime_id) as volume,
        ROUND(AVG(f.score), 2)  as avg_score,
        ROUND(AVG(f.members), 0) as avg_reach
    FROM dim_studios s
    JOIN fct_anime_stats f ON s.anime_id = f.anime_id
    GROUP BY 1
    HAVING volume > 5
    ORDER BY avg_score DESC
"""

df_studios = con.sql(queryText).df()
df_studios.head(20)

,studio,volume,avg_score,avg_reach
0,Artland,8,8.40,316487.0
1,Studio Signpost,6,8.34,77146.0
2,B.CMAY PICTURES,9,8.29,123855.0
3,Studio Ghibli,19,8.10,607446.0
4,Kinema Citrus,14,8.10,457254.0
5,Sharefun Studio,6,8.10,67557.0
6,Shuka,18,8.08,176348.0
7,SynergySP,11,8.08,107287.0
8,ufotable,29,8.07,655275.0
9,Bandai Namco Pictures,24,8.06,170521.0


I think volume > 5 skewed the results a bit. I will put 10 now.

In [18]:
queryText = """
    SELECT
        s.studio,
        COUNT(s.anime_id) as volume,
        ROUND(AVG(f.score), 2)  as avg_score,
        ROUND(AVG(f.members), 0) as avg_reach
    FROM dim_studios s
    JOIN fct_anime_stats f ON s.anime_id = f.anime_id
    GROUP BY 1
    HAVING volume > 10
    ORDER BY avg_score DESC
"""

df_studios = con.sql(queryText).df()
df_studios.head(20)

,studio,volume,avg_score,avg_reach
0,Studio Ghibli,19,8.10,607446.0
1,Kinema Citrus,14,8.10,457254.0
2,SynergySP,11,8.08,107287.0
3,Shuka,18,8.08,176348.0
4,ufotable,29,8.07,655275.0
5,Bandai Namco Pictures,24,8.06,170521.0
6,MAPPA,48,8.04,524914.0
7,Tokyo Movie Shinsha,12,8.02,143166.0
8,Madhouse,104,8.01,475997.0
9,Wit Studio,39,7.99,697450.0


It got me curious about the volume... 

In [19]:
queryText = """
    SELECT
        s.studio,
        COUNT(s.anime_id) as volume,
        ROUND(AVG(f.score), 2)  as avg_score,
        ROUND(AVG(f.members), 0) as avg_reach
    FROM dim_studios s
    JOIN fct_anime_stats f ON s.anime_id = f.anime_id
    GROUP BY 1
    HAVING volume > 10
    ORDER BY volume DESC
"""

df_studios = con.sql(queryText).df()
df_studios.head(20)

,studio,volume,avg_score,avg_reach
0,Toei Animation,122,7.75,145252.0
1,Production I.G,117,7.91,270014.0
2,Sunrise,113,7.85,168778.0
3,Madhouse,104,8.01,475997.0
4,TMS Entertainment,94,7.88,186948.0
5,A-1 Pictures,92,7.88,513354.0
6,J.C.Staff,83,7.80,398943.0
7,Studio Deen,81,7.74,191726.0
8,Kyoto Animation,69,7.94,422802.0
9,Bones,66,7.94,714321.0


In [20]:
con.close()